In [1]:
# 该notebook主要分为数据清洗或数据预处理(分别结合CDA2级教材第8章8.5逻辑回归内容)
# 导入必备库(结合CDA2级教程相关框架及思路)
from sklearn.preprocessing import StandardScaler #此处新加为解决对本数据集中连续变量进行方差膨胀因子的计算报错问题,查询资料并结合CDA2级6.3.7连续变量中心标准化思路使用特征标准化
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf
#读取数据集,测试集和训练集导入(由于kaggle数据集已经处理并分类,此处导入即可)
train=pd.read_csv(r"C:\Users\范彬\OneDrive\桌面\train.csv")
test=pd.read_csv(r"C:\Users\范彬\OneDrive\桌面\test.csv")#此处为个人数据集存放地址
print('训练集样本量 : %i \n 测试集样本量 ： %i'  %(len(train),len(test)))#个人进行数据集(训练集,测试集)体量统计
#首先,结合CDA2级教材思路,个人先对amt(金额)变量进行一元论逻辑回归模型建模(使用训练集)
lg=smf.logit("is_fraud ~ amt",train).fit()
lg.summary()
#结合CDA2级教程思路及一元逻辑回归模型结果,做出如下分析结论:
# 该模型生成信息分为二个部分,第一部分为模型的基本信息,第二部分是模型的参数估计与检验。
# 1.amt(金额)的系数为0.0028,而且p值显示系数不显著,但由于amt(金额)波动幅度相对较大,不能说明amt(金额)对欺诈影响不明显,仅显示影响幅度一般
# 2.结合CDA2级教程思路,计算OR值得e**0.0028=1.0028,即说明发生比的比值大于1，amt(金额每参加1个单位后欺诈率的可能性是原来的1.0028倍)
#结合CDA2级教程思路,多元逻辑回归中变量筛选阶段方法分为:向前回归法,向后回归法以及逐步回归法,此处借鉴书上使用的AIC准则进行的向前回归法变量筛选
# 自定义函数(复现CDA二级第八章8.5逻辑回归P219页)如下:
def forward_select(data,response):
    remaining=set(data.columns)
    remaining.remove(response)
    selected=[]
    current_score,beat_new_score=float("inf"),float("inf")
    while remaining:
        aic_with_candidates=[]
        for candidate in remaining:
            formula="{}~{}".format(response,"+".join(selected+[candidate]))
            aic=smf.logit(formula=formula,data=data).fit().aic
            aic_with_candidates.append((aic,candidate))
        aic_with_candidates.sort(reverse=True)
        best_new_score,best_candidate=aic_with_candidates.pop()
        if current_score>beat_new_score:
            remaining.remove(best_candidate)
            selected.append(best_candidate)
            current_score=best_new_score
            print("arc is {},continuing!".format(current_score))
        else:
            print("forward selection over!")
            break
    formula="{}~{}".format(response,"+".join(selected))
    print("final formula is {}".format(formula))
    model=smf.logit(formula=formula,data=data).fit()
    return model
# 以上为对CDA2级教程自定义函数的复现,但并不适用于本大体量数据集(1000000行+),以下仅供学生思路分享,大一学生学习能力有限,可能包含差错。
#接下来应用CDA2级书上使用的自定义函数即向前回归法进行对连续变量的筛选
# train_sample=train.sample(n=1000,random_state=42)#个人学生在第一次使用原数据集协方差矩阵计算,计算量过大,尝试使用取样,但出现取样is_fraud变量全部为空值，导致报错现象,故标注释化体现学生个人在学习使用CDA2级时第8章8.5.2逻辑回归模型及实现时复现线性回归法出现报错,主要原因为使用数据集体量过大及该回归模型为协方差矩阵，计算量极大,而且使用抽样法时response变量is_fraud为二分类变量,随机取样容易取空导致报错，在日后优化及版本更新过程中将使用sklearn等优化方法
#只有连续变量可以通过逐步回归进行筛选
# candidates=["amt","lat","long","merch_lat","merch_long","city_pop","trans_num","unix_time"]#沿用该数据集中的连续变量
# data_for_select=train_sample[candidates]
# lg_m1=forward_select(data=data_for_select,response="is_fraud")
# lg_m1.summary()
# print(f"原来有{len(candidates)-1}个变量")
# print(f"筛选剩下{len(lg_m1.index.tolist())}个(包含intercept 截距项)。")
# 由于数据集体量过大,使用抽样方法会导致抽取样本is_fraud类全部为0,导致直接报错

#接下来结合CDA2级教程思路,对分类变量进行显著性分析
# 补充CDA2级相关知识点，针对分类变量3种主要解决方案为:
# 1.逐一进行变量的显著性测试。
#2.使用sklearn中的决策树等方法筛选变量
# 3.使用WoE转换,通常需要进行恰当的归并或分箱。

# 接下来通过显著性测试逐一判断分类变量的显著性
class_col=["merchant","category","first","last","gender","street","city","state","zip","job"]
for i in class_col:
    tab=pd.crosstab(train[i],train["is_fraud"])
    print(i,""" p-value = %6.4f"""  %stats.chi2_contingency(tab)[1])
#得到分类变量p值均为0,属于显著，由于样本体量过大(1000000+行),本学生尝试使用抽样法进行尝试
train_sample=train.sample(n=1000,random_state=42)
for i in class_col:
    tab1=pd.crosstab(train_sample[i],train_sample["is_fraud"])
    print(i,""" p-value = %6.4f"""  %stats.chi2_contingency(tab)[1])
# 得到相同结果,在日后优化及版本更新过程中结合ai算法推荐及资料查询将使用IV值或者分箱法等优化方法

#此处第二次迭代提交时,amt(金额)变量进行一元论逻辑回归模型建模(使用训练集)被折叠,无法显示,如果查看,请查询github仓库第2次迭代记录
formula="""is_fraud~amt+C(gender)"""#此处同样进行了高基数变量的减少,由于本地环境报错及关于线性
# lg_m=smf.logit(formula=formula,data=train_sample).fit(method="bfgs",maxiter=100)#此处因报错查询使用拟牛顿法进行优化,但结果出现大量NAN值,经查询,属于准完全分离,故查询资料优化,尝试进行合并低频变量category
# cat_counts=train_sample["category"].value_counts()
# small_cats=cat_counts[cat_counts<50].index
# train_sample["category"]=train_sample["category"].replace(small_cats,"other")
# 由于处理无效,故删除大部分变量(包括category),仅保留amt和gender因变量
lg_m=smf.logit(formula=formula,data=train_sample).fit(method="bfgs",maxiter=300)
lg_m.summary()
# 此次模型拟合成功,但模型解释力不足,Pseudo R-squ=0.116,由于gernder变量的p值为0.528，大于0.5,故接下来移除gender变量并逐步增加变量
# 删除gender变量
formula1="""is_fraud~amt"""
lg_m1=smf.logit(formula=formula1,data=train_sample).fit(method="bfgs",maxiter=300)
lg_m1.summary()
# 得到模型单amt变量p值变大,为0.147，Pseudo R-squ=0.1116
# 接下来逐步增加变量
formula1="""is_fraud~amt+C(state)"""
lg_m=smf.logit(formula=formula1,data=train_sample).fit(method="bfgs",maxiter=300)
lg_m.summary()
# 出现大量NAN值,Pseudo R-squ=0.7826，说明state变量是显著性变量
# 此处沿用上面方法,单独使用category变量
formula2="""is_fraud~amt+C(category)"""
lg_m=smf.logit(formula=formula2,data=train_sample).fit(method="bfgs",maxiter=300)
lg_m.summary()
# 出现大量NAN值,Pseudo R-squ=0.7445，说明category变量是显著性变量
# 接下来尝试将state和category变量结合进行拟合
formula3="""is_fraud~amt+C(state)+C(category)"""
lg_m=smf.logit(formula=formula3,data=train_sample).fit(method="bfgs",maxiter=300)
lg_m.summary()
# 出现大量NAN值,Pseudo R-squ=0.9123，说明category和state变量对is_fraud变量是显著性影响变量,但会发现shopping_net,shopping_pops的coef值极高,属于极端值,经查询资料,本次出现了准分离问题，日后需要进行相关极端值处理。(本部分结合CDA2级教程p221页,但由于出现了大量NAN值,经查询,属于完全准分离暂时忽略,优先考虑Pseudo R-squ)


训练集样本量 : 1296675 
 测试集样本量 ： 555719
Optimization terminated successfully.
         Current function value: 0.032060
         Iterations 9
merchant  p-value = 0.0000
category  p-value = 0.0000
first  p-value = 0.0000
last  p-value = 0.0000
gender  p-value = 0.0000
street  p-value = 0.0000
city  p-value = 0.0000
state  p-value = 0.0000
zip  p-value = 0.0000
job  p-value = 0.0000
merchant  p-value = 0.0000
category  p-value = 0.0000
first  p-value = 0.0000
last  p-value = 0.0000
gender  p-value = 0.0000
street  p-value = 0.0000
city  p-value = 0.0000
state  p-value = 0.0000
zip  p-value = 0.0000
job  p-value = 0.0000
Optimization terminated successfully.
         Current function value: 0.041191
         Iterations: 33
         Function evaluations: 41
         Gradient evaluations: 39
Optimization terminated successfully.
         Current function value: 0.041393
         Iterations: 20
         Function evaluations: 26
         Gradient evaluations: 24


C:\Users\范彬\PyCharmMiscProject\.venv\Lib\site-packages\statsmodels\discrete\discrete_model.py:2385: RuntimeWarning: overflow encountered in exp
  return 1/(1+np.exp(-X))
C:\Users\范彬\PyCharmMiscProject\.venv\Lib\site-packages\statsmodels\discrete\discrete_model.py:2443: RuntimeWarning: divide by zero encountered in log
  return np.sum(np.log(self.cdf(q * linpred)))
C:\Users\范彬\PyCharmMiscProject\.venv\Lib\site-packages\statsmodels\discrete\discrete_model.py:2385: RuntimeWarning: overflow encountered in exp
  return 1/(1+np.exp(-X))
C:\Users\范彬\PyCharmMiscProject\.venv\Lib\site-packages\statsmodels\discrete\discrete_model.py:2443: RuntimeWarning: divide by zero encountered in log
  return np.sum(np.log(self.cdf(q * linpred)))
C:\Users\范彬\PyCharmMiscProject\.venv\Lib\site-packages\statsmodels\discrete\discrete_model.py:2385: RuntimeWarning: overflow encountered in exp
  return 1/(1+np.exp(-X))
C:\Users\范彬\PyCharmMiscProject\.venv\Lib\site-packages\statsmodels\discrete\discrete_model.py:24

Optimization terminated successfully.
         Current function value: 0.010130
         Iterations: 125
         Function evaluations: 135
         Gradient evaluations: 133
Optimization terminated successfully.
         Current function value: 0.011903
         Iterations: 85
         Function evaluations: 92
         Gradient evaluations: 90


C:\Users\范彬\PyCharmMiscProject\.venv\Lib\site-packages\statsmodels\base\model.py:595: HessianInversionWarning: Inverting hessian failed, no bse or cov_params available
  warnings.warn('Inverting hessian failed, no bse or cov_params '
C:\Users\范彬\PyCharmMiscProject\.venv\Lib\site-packages\statsmodels\discrete\discrete_model.py:2385: RuntimeWarning: overflow encountered in exp
  return 1/(1+np.exp(-X))
C:\Users\范彬\PyCharmMiscProject\.venv\Lib\site-packages\statsmodels\discrete\discrete_model.py:2443: RuntimeWarning: divide by zero encountered in log
  return np.sum(np.log(self.cdf(q * linpred)))
C:\Users\范彬\PyCharmMiscProject\.venv\Lib\site-packages\statsmodels\base\model.py:595: HessianInversionWarning: Inverting hessian failed, no bse or cov_params available
  warnings.warn('Inverting hessian failed, no bse or cov_params '
C:\Users\范彬\PyCharmMiscProject\.venv\Lib\site-packages\statsmodels\discrete\discrete_model.py:2385: RuntimeWarning: overflow encountered in exp
  return 1/(1+np.exp(

Optimization terminated successfully.
         Current function value: 0.004088
         Iterations: 114
         Function evaluations: 123
         Gradient evaluations: 121


C:\Users\范彬\PyCharmMiscProject\.venv\Lib\site-packages\statsmodels\base\model.py:595: HessianInversionWarning: Inverting hessian failed, no bse or cov_params available
  warnings.warn('Inverting hessian failed, no bse or cov_params '


<class 'statsmodels.iolib.summary.Summary'>
"""
                           Logit Regression Results                           
==============================================================================
Dep. Variable:               is_fraud   No. Observations:                 1000
Model:                          Logit   Df Residuals:                      937
Method:                           MLE   Df Model:                           62
Date:                Fri, 27 Mar 2026   Pseudo R-squ.:                  0.9123
Time:                        19:52:27   Log-Likelihood:                -4.0884
converged:                       True   LL-Null:                       -46.594
Covariance Type:            nonrobust   LLR p-value:                   0.02785
=================================================================================================
                                    coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------------------
Intercept                      -103.5193        nan        nan        nan         nan         nan
C(state)[T.AL]                   -9.3424        nan        nan        nan         nan         nan
C(state)[T.AR]                   -8.6224        nan        nan        nan         nan         nan
C(state)[T.AZ]                   -2.6110        nan        nan        nan         nan         nan
C(state)[T.CA]                  -16.0571        nan        nan        nan         nan         nan
C(state)[T.CO]                   -3.4550        nan        nan        nan         nan         nan
C(state)[T.CT]                   -2.0835        nan        nan        nan         nan         nan
C(state)[T.DC]                   -0.2901        nan        nan        nan         nan         nan
C(state)[T.FL]                   -9.3599        nan        nan        nan         nan         nan
C(state)[T.GA]                  -53.7974        nan        nan        nan         nan         nan
C(state)[T.HI]                   -0.6105        nan        nan        nan         nan         nan
C(state)[T.IA]                   -3.9630        nan        nan        nan         nan         nan
C(state)[T.ID]                   -2.1062        nan        nan        nan         nan         nan
C(state)[T.IL]                  -12.2024        nan        nan        nan         nan         nan
C(state)[T.IN]                   -6.8784        nan        nan        nan         nan         nan
C(state)[T.KS]                   36.9434        nan        nan        nan         nan         nan
C(state)[T.KY]                   -6.2268        nan        nan        nan         nan         nan
C(state)[T.LA]                   -6.8354        nan        nan        nan         nan         nan
C(state)[T.MA]                   -2.4507        nan        nan        nan         nan         nan
C(state)[T.MD]                   -6.2985        nan        nan        nan         nan         nan
C(state)[T.ME]                   -2.0734        nan        nan        nan         nan         nan
C(state)[T.MI]                   -8.6674        nan        nan        nan         nan         nan
C(state)[T.MN]                   -6.7151        nan        nan        nan         nan         nan
C(state)[T.MO]                  -10.3996        nan        nan        nan         nan         nan
C(state)[T.MS]                   -4.7494        nan        nan        nan         nan         nan
C(state)[T.MT]                   -3.2020        nan        nan        nan         nan         nan
C(state)[T.NC]                   -6.4499        nan        nan        nan         nan         nan
C(state)[T.ND]                   -3.1954        nan        nan        nan         nan         nan
C(state)[T.NE]                   -5.1193        nan        nan        nan         nan         nan
C(state)[T.NH]                   -2.4701        nan        nan        nan         nan         nan
C(state

In [3]:
# 结合CDA二级教程思路及框架(具体为8.5.2逻辑回归模型最后部分中),接下来针对自变量的多重共线性使用方差膨胀因子进行分析
# 方差膨胀因子检验(自定义函数借鉴引用CDA2级教程P222,个人进行相关注释及原理补充)
def vif(df,col_i):
    from statsmodels.formula.api import ols
    cols=list(df.columns)#取出所有类名
    cols.remove(col_i)#移除要计算VIF的变量
    cols_noti=cols#剩余类作为自变量
    formula=col_i+"~"+"+".join(cols_noti)#构建回归公式 当前类~其他列
    r2=ols(formula,df).fit().rsquared##拟合线性回归并获取R**2
    return 1/(1-r2)#代入公式VIF=1/(1-R**2),计算VIF
# 经查询资料,补充方差膨胀因子(VIF)压原理:
# 将每个自变量依次作为因变量,对其他自变量作线性回归,得到R**2,再用公式VIF=1/(1-R**2)进行计算
# VIF阀值参考:
# VIF<5 :共线性极弱,可忽略
# 5<=VIF<=10:存在一定共线性,需要关注
# VIF》10:存在严重共线性,必须处理(如删除变量,PCA降维)

# 继续根据CDA2级教程思路运用自定义函数,对本数据集中连续变量进行方差膨胀因子的计算
# 定义连续变量列表
candidates=["amt","lat","long","merch_long","merch_lat","city_pop"]
# 提取列表目标变量,删除目标变量is_fraud
# exog=train[candidates].drop(["is-fraud"],axis=1)
# 第一次运行上行,由于is_fraud列本身不在category里,故出现报错,经查询资料后进行相关修改
exog=train[candidates]
for i in exog.columns:
    print(i,"/t",vif(df=exog,col_i=i))
# 运行后出现了大量报错提醒,查询资料并结合CDA2级6.3.7连续变量中心标准化思路使用特征标准化
scaler=StandardScaler()
exog_scaled=pd.DataFrame(scaler.fit_transform(exog.values),columns=exog.columns)
# 再次计算VIF(方差膨胀因子)
print(i,"/t",vif(df=exog,col_i=i))
# 根据生成的分析结果及查询资料得:
# lat，long,merch_long,merch_lat值过大,说明具有严重的多重共线性问题
# amt,city_pop的VIF值小于5,共线性极小,可忽略

amt /t 1.000035813299742
lat /t 78.30684846647917
long /t 568.1653056038406
merch_long /t 568.1628567138872
merch_lat /t 78.28270409648616
city_pop /t 1.028090354811022
city_pop /t 1.028090354811022
